In [1]:
! pip install ramanujantools==0.0.7

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 77.2 MB/s eta 0:00:00


In [2]:
import sympy as sp
from sympy.abc import n

from ramanujantools import Matrix, LinearRecurrence, Position
from ramanujantools.cmf import CMF, MeijerG

a0, a1, a2, a3 = sp.symbols("a:4")
b0, b1, b2, b3, b4 = sp.symbols("b:5")

# Our setting - the Meijer G-function with m=q=3, n=p=2.
cmf = MeijerG(3, 2, 2, 3, 1)

In [3]:
# Our (1, 1; 1+n, 1+n, 1+n) parameterization is start + n * trajectory
trajectory = Position({a0: 0, a1: 0, b0: 1, b1: 1, b2: 1})
start = Position({a0: 1, a1: 1, b0: 1, b1: 1, b2: 1})

# m here is M(n) in the solution; m_T here is M-hat.
m = cmf.trajectory_matrix(trajectory, start)
m_T = m.inverse().transpose().as_polynomial()

In [4]:
# The initial values from our identity of the Meijer G-function.
# First row represents θH(1), second row represents H(1) - θH(1) + θ^2H(1).
iv = Matrix([[0, 1, 0], [1, -1 ,1]])
display(iv)

Matrix([
[0,  1, 0],
[1, -1, 1]])

In [5]:
# Sanity check: the ungauged dual system approximates Euler's constant.
l = m_T.limit({n: 1}, 100, {n: 0}, iv)
print(l.as_float())

0.5772156649015328606065120900824024310351


In [6]:
# Our primal system M(n)
display(m)

Matrix([
[0,    (n + 1)**3,   (n + 1)**3*(3*n + 4)],
[0, -3*(n + 1)**2, (-8*n - 11)*(n + 1)**2],
[1,       3*n + 3,      6*n**2 + 12*n + 5]])

In [7]:
# Converting it to a recurrence using CVM - part 1: finding the gauge matrix U(n)
U = m.companion_coboundary_matrix()
display(U)

Matrix([
[1, 0, 3*n**4 + 13*n**3 + 21*n**2 + 15*n + 4],
[0, 0,         -8*n**3 - 27*n**2 - 30*n - 11],
[0, 1,                     6*n**2 + 12*n + 5]])

In [8]:
# Converting it to a recurrence using CVM - part 2: applying the gauge
forward_recurrence = LinearRecurrence(m.coboundary(U))
display(forward_recurrence)

LinearRecurrence([8*n + 11, -24*n**3 - 105*n**2 - 124*n - 25, -(-24*n**5 - 241*n**4 - 964*n**3 - 1920*n**2 - 1904*n - 752), -8*n**7 - 83*n**6 - 360*n**5 - 846*n**4 - 1164*n**3 - 939*n**2 - 412*n - 76])

In [9]:
# The asymptotics extracted for this specific parameterization
solutions = forward_recurrence.asymptotics()
for solution in solutions:
  display(solution)

n**(16/3)*exp(-I*n**(1/3)*(3*sqrt(3)*n**(1/3) + 3*I*n**(1/3) + sqrt(3) - I)/2)*factorial(n)**2

n**(16/3)*exp(I*n**(1/3)*(3*sqrt(3)*n**(1/3) - 3*I*n**(1/3) + sqrt(3) + I)/2)*factorial(n)**2

n**(16/3)*exp(-n**(1/3)*(3*n**(1/3) - 1))*factorial(n)**2

In [10]:
# Sanity: numerically verify that our meijer g is the recessive solution
from sympy import meijerg

def H(n):
  return meijerg(((1, 1), ()), ((n + 1, n + 1, n + 1), ()), 1)

N = 2000

for solution in solutions:
  # plus three due to convention - the recurrence looks backwards
  display(sp.re(solution.subs({n: N}) / H(N+3).evalf()).evalf())

-4.77405190967005e+301

-4.77405190967005e+301

0.641277643288365

In [11]:
# Our m hat dual system
display(m_T)

Matrix([
[              6*n**2 + 21*n + 18,         -8*n - 11,     3],
[     3*n**3 + 12*n**2 + 16*n + 7, -3*n**2 - 7*n - 4, n + 1],
[n**4 + 4*n**3 + 6*n**2 + 4*n + 1,                 0,     0]])

In [12]:
# Converting the dual system now to cvm
U = m_T.companion_coboundary_matrix()
display(U)

Matrix([
[1,               6*n**2 + 21*n + 18,                    15*n**4 + 147*n**3 + 520*n**2 + 792*n + 440],
[0,      3*n**3 + 12*n**2 + 16*n + 7,          10*n**5 + 96*n**4 + 353*n**3 + 625*n**2 + 537*n + 179],
[0, n**4 + 4*n**3 + 6*n**2 + 4*n + 1, 6*n**6 + 57*n**5 + 213*n**4 + 402*n**3 + 408*n**2 + 213*n + 45]])

In [13]:
# Our target recurrence
r = LinearRecurrence(m_T.coboundary(U))
display(r)

LinearRecurrence([-8*n**3 - 51*n**2 - 105*n - 68, 24*n**5 + 337*n**4 + 1833*n**3 + 4818*n**2 + 6092*n + 2928, -24*n**7 - 393*n**6 - 2659*n**5 - 9542*n**4 - 19305*n**3 - 21367*n**2 - 11150*n - 1608, 8*n**9 + 171*n**8 + 1603*n**7 + 8645*n**6 + 29541*n**5 + 66280*n**4 + 97544*n**3 + 90672*n**2 + 48208*n + 11136])

In [14]:
# The gauged initial values
iv_gauged = iv * U.subs({n: 0})
display(iv_gauged)

Matrix([
[0,  7, 179],
[1, 12, 306]])

In [15]:
l_recurrence = r.limit(100, 0, iv_gauged)
print(l_recurrence.as_float())

0.57721566490153286060651209008240243104132
